In [1]:
import numpy as np
import pandas as pd
import geopandas as gpd
from scipy.spatial.distance import pdist, squareform
from scipy.sparse import csr_matrix, lil_matrix
from pyproj import Transformer
from sklearn.preprocessing import StandardScaler
import pyreadr 
from pathlib import Path
import os

os.chdir(Path.cwd().parent)


In [23]:
import numpy as np
import pandas as pd
from tqdm import tqdm
from scipy.sparse import csr_matrix, coo_matrix, hstack, bmat, diags
from scipy.sparse.linalg import splu
from sksparse.cholmod import cholesky
from polyagamma import random_polyagamma
from scipy.spatial.distance import pdist, squareform
import geopandas as gpd
import pyreadr
from sklearn.preprocessing import StandardScaler

# ============================================================
# Load data
# ============================================================

no_nbs = np.array([57,170,236,269,343,685,946,947,989,1037,1084,1090,1109,1118,1127,1176,1203]) - 1

snow_temp_full = pyreadr.read_r("snow_temp_full.Rda")
snow_temp_full = list(snow_temp_full.values())[0].reset_index(drop=True)
temp = snow_temp_full.iloc[:, 2:]         # drop lon/lat
temp1 = temp.drop(index=no_nbs)
temp2 = temp.iloc[no_nbs]
temp_aligned = pd.concat([temp1, temp2], axis=0).reset_index(drop=True)
temp_aligned = temp_aligned.to_numpy()     # shape = (S, TT)

snow_cleaned_full = pyreadr.read_r("snow_cleaned_full.Rda")
snow_cleaned_full = list(snow_cleaned_full.values())[0]

df1 = snow_cleaned_full.drop(index=no_nbs)
df2 = snow_cleaned_full.iloc[no_nbs]
all_y = pd.concat([df1, df2], axis=0).reset_index(drop=True)

y = all_y.iloc[:, 2:].to_numpy()
coords = all_y.iloc[:, :2].rename(columns={0: "LON", 1: "LAT"}).to_numpy()

# ============================================================
# Elevation – match R scale() exactly (mean + sd(ddof=1))
# ============================================================

curr_elev = pd.read_csv("curr_elev.csv").iloc[:, 3].to_numpy()
nnbs = pd.read_csv("nnbs_elev.csv", sep="\t").dropna(axis=1, how='all')
nnbs_elev = nnbs.iloc[:, -1].astype(float).to_numpy()

elev_raw = np.concatenate([curr_elev, nnbs_elev])
elev_mean = elev_raw.mean()
elev_sd   = elev_raw.std(ddof=1)   # R scale() uses sample SD
elev = (elev_raw - elev_mean) / elev_sd

# ============================================================
# Latitude – also match R scale() exactly
# ============================================================

lat_raw = coords[:,1]
lat_mean = lat_raw.mean()
lat_sd   = lat_raw.std(ddof=1)
lats = (lat_raw - lat_mean) / lat_sd

# ============================================================
# Spatial projection + rotation (unchanged)
# ============================================================

gdf = gpd.GeoDataFrame(
    geometry=gpd.points_from_xy(coords[:,0], coords[:,1]),
    crs="EPSG:4326"
)
gdf_aeqd = gdf.to_crs("+proj=aeqd +lat_0=90 +lon_0=-100")
xy = np.vstack([gdf_aeqd.geometry.x, gdf_aeqd.geometry.y]).T

dif = xy[1] - xy[2]
theta = np.arctan2(dif[1], dif[0])

Rmat = np.array([[np.cos(theta), -np.sin(theta)],
                 [np.sin(theta),  np.cos(theta)]])

rotated = (xy @ Rmat.T) / 1e6

Distances = squareform(pdist(rotated))
Omg = (Distances <= 0.22).astype(int)
np.fill_diagonal(Omg, 0)
Omg = csr_matrix(Omg)

D = csr_matrix(np.diag(np.array(Omg.sum(axis=1)).flatten()))
prec = D - Omg

S, TT = y.shape
period = 52

# ============================================================
# Build logistic regression data set
# ============================================================

loc = np.where(y[:, :-1] == 0)
pairs = np.column_stack(loc)
order = np.lexsort((pairs[:,0], pairs[:,1]))
pairs_R = pairs[order]
pairs_R[:,1] += 1

row_idx = pairs_R[:,0]
time_idx = pairs_R[:,1] - 1
N = len(row_idx)

next_y = y[pairs_R[:,0], pairs_R[:,1]]

t = time_idx + 1

temp_design_raw = temp_aligned[row_idx, time_idx]
temp_mean = temp_design_raw.mean()
temp_sd   = temp_design_raw.std()    # ddof=0 is OK
temp_scaled = (temp_design_raw - temp_mean) / temp_sd

# ============================================================
# Covariates – match R exactly (8 columns)
# ============================================================

covariates = np.column_stack([
    np.ones(N), np.ones(N),
    np.cos(2*np.pi*t/period), np.cos(2*np.pi*t/period),
    np.sin(2*np.pi*t/period), np.sin(2*np.pi*t/period),
    t, t,
    temp_scaled, temp_scaled
])

K = covariates.shape[1]
print("Covariates shape:", covariates.shape)

# ============================================================
# Construct design_loc – verified identical to R version
# ============================================================

rows, cols, vals = [], [], []

for i in tqdm(range(N), desc="Building design_loc"):
    loc_i = row_idx[i]
    cov = covariates[i]
    for k in range(K):
        rows.append(i)
        cols.append(loc_i + k*S)
        vals.append(cov[k])

design_loc = coo_matrix((vals, (rows, cols)), shape=(N, S*K)).tocsr()

# ============================================================
# Add elevation and latitude (order MUST match R: elev → lat)
# ============================================================

elev_col = csr_matrix(elev[row_idx]).reshape(N,1)
lat_col  = csr_matrix(lats[row_idx]).reshape(N,1)

design_mat = hstack([design_loc, elev_col, lat_col]).tocsr()

theta_dim = K*S + 2   # last 2 = elev, lat

# ============================================================
# MCMC SETTINGS
# ============================================================

burn = 1000  
thin = 5  
tot_save = 1000 

total_iters = burn + tot_save * thin

all_theta = np.zeros((theta_dim, tot_save))
all_tau   = np.zeros((K, tot_save))

curr_theta = np.zeros(theta_dim)
curr_tau   = np.ones(K)

a_tau = 0.001
b_tau = 0.001

kappa = next_y - 0.5

curr_idx = 0
save_idx = 0

# ============================================================
# MCMC LOOP with burn & thin
# ============================================================

for it in tqdm(range(total_iters), desc="MCMC"):

    curr_idx += 1

    # 1. PG augmentation
    phi = design_mat.dot(curr_theta)
    omega = random_polyagamma(1, phi, size=N)

    # 2. Build precision block structure
    block_list = []
    for j in range(K):
        if j % 2 == 0:
            block_list.append((1/curr_tau[j]) * prec)   # CAR
        else:
            block_list.append((1/curr_tau[j]) * diags(np.ones(S)))  # IID

    # last block = elevation + lat
    block_list.append((1/100) * diags(np.ones(2)))

    blocks = []
    for i in range(K+1):
        row = []
        for j in range(K+1):
            if i == j:
                row.append(block_list[i])
            else:
                row.append(None)
        blocks.append(row)

    curr_prec = bmat(blocks, format="csr")

    # 3. Posterior precision
    xtOmega = design_mat.T.multiply(omega)
    pos_prec = xtOmega.dot(design_mat) + curr_prec

    # 4. Cholesky
    factor = cholesky(pos_prec)

    mu = factor.solve_A(design_mat.T.dot(kappa))

    # theta
    eps = np.random.randn(theta_dim)
    curr_theta = mu + factor.solve_A(eps)

    # 5. Update taus
    for j in range(K):
        sl = slice(j*S, (j+1)*S)
        beta = curr_theta[sl]

        if j % 2 == 0:
            quad = beta @ prec.dot(beta)
        else:
            quad = beta @ beta

        curr_tau[j] = 1 / np.random.gamma(a_tau + S/2, 1/(b_tau + quad/2))

    # =====================================================
    # Save if beyond burn and it % thin == 0
    # =====================================================
    if it >= burn and ((it - burn) % thin == 0):
        all_theta[:, save_idx] = curr_theta
        all_tau[:, save_idx] = curr_tau
        save_idx += 1

        if save_idx >= tot_save:
            break

# ============================================================
# Save output
# ============================================================

np.savez_compressed("bym++01.npz",
                    all_theta=all_theta,
                    all_tau=all_tau)


Covariates shape: (2800090, 10)


MCMC:   0%|          | 0/6000 [00:00<?, ?it/s]C:\Users\lix23\AppData\Local\Temp\ipykernel_37024\1024965270.py:218: CholmodTypeConversionWarning: converting matrix of class csr_matrix to CSC format
  factor = cholesky(pos_prec)
MCMC: 100%|█████████▉| 5995/6000 [7:29:10<00:22,  4.50s/it]  


In [24]:
import numpy as np
import pandas as pd
from tqdm import tqdm
from scipy.sparse import csr_matrix, coo_matrix, hstack, bmat, diags
from scipy.sparse.linalg import splu
from sksparse.cholmod import cholesky
from polyagamma import random_polyagamma
from scipy.spatial.distance import pdist, squareform
import geopandas as gpd
import pyreadr
from sklearn.preprocessing import StandardScaler

# ============================================================
# Load data (same as p01)
# ============================================================

no_nbs = np.array([57,170,236,269,343,685,946,947,989,1037,1084,1090,1109,1118,1127,1176,1203]) - 1

snow_temp_full = pyreadr.read_r("snow_temp_full.Rda")
snow_temp_full = list(snow_temp_full.values())[0].reset_index(drop=True)
temp = snow_temp_full.iloc[:, 2:]
temp1 = temp.drop(index=no_nbs)
temp2 = temp.iloc[no_nbs]
temp_aligned = pd.concat([temp1, temp2], axis=0).reset_index(drop=True).to_numpy()

snow_cleaned_full = pyreadr.read_r("snow_cleaned_full.Rda")
snow_cleaned_full = list(snow_cleaned_full.values())[0]

df1 = snow_cleaned_full.drop(index=no_nbs)
df2 = snow_cleaned_full.iloc[no_nbs]
all_y = pd.concat([df1, df2], axis=0).reset_index(drop=True)

y = all_y.iloc[:, 2:].to_numpy()
coords = all_y.iloc[:, :2].rename(columns={0: "LON", 1: "LAT"}).to_numpy()

# ============================================================
# Elevation / Latitude (same as p01)
# ============================================================
curr_elev = pd.read_csv("curr_elev.csv").iloc[:, 3].to_numpy()
nnbs = pd.read_csv("nnbs_elev.csv", sep="\t").dropna(axis=1, how='all')
nnbs_elev = nnbs.iloc[:, -1].astype(float).to_numpy()

elev_raw = np.concatenate([curr_elev, nnbs_elev])
elev = (elev_raw - elev_raw.mean()) / elev_raw.std(ddof=1)

lat_raw = coords[:,1]
lats = (lat_raw - lat_raw.mean()) / lat_raw.std(ddof=1)

# ============================================================
# Spatial adjacency (same as p01)
# ============================================================
gdf = gpd.GeoDataFrame(
    geometry=gpd.points_from_xy(coords[:,0], coords[:,1]),
    crs="EPSG:4326"
)
gdf_aeqd = gdf.to_crs("+proj=aeqd +lat_0=90 +lon_0=-100")
xy = np.vstack([gdf_aeqd.geometry.x, gdf_aeqd.geometry.y]).T

dif = xy[1] - xy[2]
theta = np.arctan2(dif[1], dif[0])
Rmat = np.array([[np.cos(theta), -np.sin(theta)],
                 [np.sin(theta),  np.cos(theta)]])
rotated = (xy @ Rmat.T) / 1e6

Distances = squareform(pdist(rotated))
Omg = (Distances <= 0.22).astype(int)
np.fill_diagonal(Omg, 0)
Omg = csr_matrix(Omg)
D = csr_matrix(np.diag(np.array(Omg.sum(axis=1)).flatten()))
prec = D - Omg

S, TT = y.shape
period = 52

# ============================================================
# Build logistic regression data set (THIS IS WHAT changes)
# ============================================================

# p10 condition: y_t = 1
loc = np.where(y[:, :-1] == 1)
pairs = np.column_stack(loc)
order = np.lexsort((pairs[:,0], pairs[:,1]))
pairs_R = pairs[order]
pairs_R[:,1] += 1

row_idx = pairs_R[:,0]
time_idx = pairs_R[:,1] - 1
N = len(row_idx)

next_y = y[pairs_R[:,0], pairs_R[:,1]]

# --- p10 event definition ---
event = 1 - next_y            # melting: 1→0 gives event = 1
kappa = event - 0.5           # canonical PG form

# ============================================================
# Covariates (same as p01)
# ============================================================

t = time_idx + 1

temp_design_raw = temp_aligned[row_idx, time_idx]
temp_scaled = (temp_design_raw - temp_design_raw.mean()) / temp_design_raw.std()

covariates = np.column_stack([
    np.ones(N), np.ones(N),
    np.cos(2*np.pi*t/period), np.cos(2*np.pi*t/period),
    np.sin(2*np.pi*t/period), np.sin(2*np.pi*t/period),
    t, t,
    temp_scaled, temp_scaled
])

K = covariates.shape[1]

# ============================================================
# Construct design matrix (same as p01)
# ============================================================

rows, cols, vals = [], [], []
for i in tqdm(range(N), desc="Building design_loc"):
    loc_i = row_idx[i]
    cov = covariates[i]
    for k in range(K):
        rows.append(i)
        cols.append(loc_i + k*S)
        vals.append(cov[k])

design_loc = coo_matrix((vals, (rows, cols)), shape=(N, S*K)).tocsr()

elev_col = csr_matrix(elev[row_idx]).reshape(N,1)
lat_col  = csr_matrix(lats[row_idx]).reshape(N,1)

design_mat = hstack([design_loc, elev_col, lat_col]).tocsr()

theta_dim = K*S + 2

# ============================================================
# MCMC SETTINGS  (same as p01)
# ============================================================

burn = 1000
thin = 5
tot_save = 1000
total_iters = burn + tot_save * thin

all_theta = np.zeros((theta_dim, tot_save))
all_tau   = np.zeros((K, tot_save))

curr_theta = np.zeros(theta_dim)
curr_tau   = np.ones(K)

a_tau = 0.001
b_tau = 0.001

curr_idx = 0
save_idx = 0

# ============================================================
# MCMC LOOP (identical to p01 except kappa)
# ============================================================

for it in tqdm(range(total_iters), desc="MCMC"):

    curr_idx += 1

    phi = design_mat.dot(curr_theta)
    omega = random_polyagamma(1, phi, size=N)

    block_list = []
    for j in range(K):
        if j % 2 == 0:
            block_list.append((1/curr_tau[j]) * prec)
        else:
            block_list.append((1/curr_tau[j]) * diags(np.ones(S)))
    block_list.append((1/100) * diags(np.ones(2)))

    blocks = []
    for i in range(K+1):
        row = []
        for j in range(K+1):
            row.append(block_list[i] if i == j else None)
        blocks.append(row)

    curr_prec = bmat(blocks, format="csr")

    xtOmega = design_mat.T.multiply(omega)
    pos_prec = xtOmega.dot(design_mat) + curr_prec
    factor = cholesky(pos_prec)

    mu = factor.solve_A(design_mat.T.dot(kappa))

    eps = np.random.randn(theta_dim)
    curr_theta = mu + factor.solve_A(eps)

    for j in range(K):
        sl = slice(j*S, (j+1)*S)
        beta = curr_theta[sl]
        quad = beta @ prec.dot(beta) if j % 2 == 0 else beta @ beta
        curr_tau[j] = 1 / np.random.gamma(a_tau + S/2, 1/(b_tau + quad/2))

    if it >= burn and ((it - burn) % thin == 0):
        all_theta[:, save_idx] = curr_theta
        all_tau[:, save_idx] = curr_tau
        save_idx += 1
        if save_idx >= tot_save:
            break

np.savez_compressed("bym++10.npz",
                    all_theta=all_theta,
                    all_tau=all_tau)


MCMC:   0%|          | 0/6000 [00:00<?, ?it/s]C:\Users\lix23\AppData\Local\Temp\ipykernel_37024\3164645372.py:188: CholmodTypeConversionWarning: converting matrix of class csr_matrix to CSC format
  factor = cholesky(pos_prec)
MCMC: 100%|█████████▉| 5995/6000 [4:09:21<00:12,  2.50s/it]  
